In [ ]:
import io
import zipfile
import requests
import frontmatter
import re
from tqdm.auto import tqdm
from openai import OpenAI

def read_repo_data(repo_owner,repo_name):
    """
    Download and parse all markdown files from github repository.
    Args:
        repo_owner: Github username and organization
        repo_name: Repository name
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com'
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)

    if resp.status_code != 200:
        raise Exception(f"Failed to download repository : {resp.status_code}")
    repository_data = []
    zf=zipfile.ZipFile(io.BytesIO(resp.content))

    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not(filename_lower.endswith('md') 
            or(filename_lower.endswith('mdx'))):
            continue;

        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8',errors='ignore')
                post=frontmatter.loads(content)
                data=post.to_dict()
                data['filename']=filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename} : {e}")
            continue
    zf.close()
    return repository_data
    
evidentlyai_docs = read_repo_data('evidentlyai','docs')
print(f"Evidently ai docs : {len(evidentlyai_docs)}")

def sliding_window(seq,size,step):
    """ Simple chunking """
    if size<=0 or step<=0:
        raise ValueError("size and step must be positive")
    if size<=step:
        raise ValueError("size should be more than step")
    n = len(seq)
    result = []
    for i in range(0,n,step):
        chunk=seq[i:i+size]
        result.append({'start':i,'chunk':chunk})
        if i+size>=n:
            break
    return result

# # Start - Simple chunking 
# evidently_chunks = []
# for doc in evidentlyai_docs:
   
#     doc_copy=doc.copy()
#     doc_content=doc_copy.pop('content')
#     chunks=sliding_window(doc_content,2000,1000)
#     for chunk in chunks:
#         chunk.update(doc_copy)
#     evidently_chunks.extend(chunks)
# print(len(evidently_chunks)) 
# #End - Simple chunking 

# Splitting by sections
def split_markdown_by_section(text,level=2):
    """
    Split markdown text by a specific header level.

    :param text: markdown text as string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    #This regex matches markdown headers
    #For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    #Split and keep the headers
    parts = pattern.split(text)

    sections = []
    for i in range(1, len(parts), 3):
        #We step by 3 because regex.split with capturing groups returns:
        #[before_match,group1,group2,after_match,...]
        #Here group1 is "## ", group 2 is header text
        header=parts[i]+parts[i+1]
        header=header.strip()

        #Get content after this header
        content=""
        if(i+2<len(parts)):
            content=parts[i+2].strip()

        if content:
            section=f'{header}\n\n{content}'
        else:
            section=header
        sections.append(section)
    return sections

#Splitting by sections
# evidently_chunks = []
# for doc in evidentlyai_docs:
#     doc_copy = doc.copy()
#     doc_content = doc.pop('content')
#     sections=split_markdown_by_section(doc_content, level=2)
#     for section in sections:
#         section_doc = doc.copy()
#         section_doc['section']=section
#         evidently_chunks.append(section_doc)
    
# Intelligent chunking with LLM
prompt_template = """
Split the provided document into logical sections that make sense for a Q&A system.
Each section should be self contained and cover a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>
Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---

""".strip()
# Used llama3.2 as don't have open ai api key
def llm(prompt, model='llama3.2:latest'):
    client=OpenAI(base_url="http://localhost:11434/v1",api_key="ollama")
    response=client.chat.completions.create(model="llama3.2:latest",
                                            messages=[{"role":"user","content":prompt}])
    return response.choices[0].message.content
def intelligent_chunking(text):
    prompt=prompt_template.format(document=text)
    response=llm(prompt)
    sections=response.split("---")
    sections=[s.strip() for s in sections if s.strip()]
    return sections

# Apply intelligent chunking ot every document

evidently_chunks=[]
for doc in tqdm(evidentlyai_docs):
    doc_copy=doc.copy()
    doc_content=doc_copy.pop('content')
    if doc_content:
        sections=intelligent_chunking(doc_content)
        for section in sections:
            section_doc=doc_copy.copy()
            section_doc['section']=section
            evidently_chunks.append(section_doc)

        

Evidently ai docs : 95


  0%|          | 0/95 [00:00<?, ?it/s]

DOC CONTENT :  <Note>
  If you're not looking to build API reference documentation, you can delete
  this section by removing the api-reference folder.
</Note>

## Welcome

There are two ways to build API documentation: [OpenAPI](https://mintlify.com/docs/api-playground/openapi/setup) and [MDX components](https://mintlify.com/docs/api-playground/mdx/configuration). For the starter kit, we are using the following OpenAPI specification.

<Card
  title="Plant Store Endpoints"
  icon="leaf"
  href="https://github.com/mintlify/starter/blob/main/api-reference/openapi.json"
>
  View the OpenAPI specification file
</Card>

## Authentication

All API endpoints are authenticated using Bearer tokens and picked up from the specification file.

```json
"security": [
  {
    "bearerAuth": []
  }
]
```
PROMPT Split the provided document into logical sections that make sense for a Q&A system.
Each section should be self contained and cover a specific topic or concept.

<DOCUMENT>
<Note>
  If you're no